# 米国株AIレポート MVP（Colabで1回動かす版）

このノートブックは、以下を1回で実行します。

1. Google Driveをマウント
2. 必要ライブラリをインストール
3. YouTube検索・RSSから公開情報を取得
4. Jim Cramer / Ray Dalio / Scott Galloway 関連情報を整理
5. 米国株ティッカーを簡易抽出
6. yfinanceで市場データを取得
7. PDFレポートをGoogle Driveへ保存
8. Gmail SMTPで指定メールアドレスへ送信

注意：
- 初期MVPなので、外部LLM APIは使いません。
- YouTube音声のダウンロードは行いません。字幕・説明文・タイトルを優先します。
- 本レポートは投資助言ではなく、公開情報を整理した調査資料です。

In [ ]:
# セル1：Google Driveをマウントし、必要ライブラリを入れます
from google.colab import drive
drive.mount('/content/drive')

!apt-get -qq update > /dev/null
!apt-get -qq install -y fonts-noto-cjk > /dev/null
!pip -q install yfinance feedparser beautifulsoup4 requests reportlab pandas yt-dlp

print("準備完了：Driveマウントとライブラリ導入が終わりました。")

In [ ]:
# セル2：あなたの情報を入力します
# Gmailの通常パスワードではなく「アプリパスワード」を入れてください。
# 入力されたパスワードは画面に表示されません。

import getpass
import os
from datetime import datetime

DRIVE_ROOT = "/content/drive/MyDrive/ai-investment-agent"
REPORT_DIR = os.path.join(DRIVE_ROOT, "reports")
LOG_DIR = os.path.join(DRIVE_ROOT, "logs")
DATA_DIR = os.path.join(DRIVE_ROOT, "data")

for p in [DRIVE_ROOT, REPORT_DIR, LOG_DIR, DATA_DIR]:
    os.makedirs(p, exist_ok=True)

print("保存先:", DRIVE_ROOT)

EMAIL_TO = input("レポートを受け取るメールアドレスを入力してください: ").strip()
SMTP_USER = input("送信用Gmailアドレスを入力してください: ").strip()
EMAIL_FROM = SMTP_USER
SMTP_PASSWORD = getpass.getpass("Gmailアプリパスワード16桁を入力してください: ").strip()

send_choice = input("PDFをメール送信しますか？ y/n [y]: ").strip().lower()
SEND_EMAIL = (send_choice != "n")

print("設定完了")
print("EMAIL_TO:", EMAIL_TO)
print("SMTP_USER:", SMTP_USER)
print("SEND_EMAIL:", SEND_EMAIL)

In [ ]:
# セル3：MVP本体。ここを実行すると、情報取得→分析→PDF作成→メール送信まで行います。

import os
import re
import json
import html
import math
import time
import smtplib
import logging
import traceback
from email.message import EmailMessage
from datetime import datetime, timezone
from collections import defaultdict
from xml.sax.saxutils import escape

import requests
import feedparser
import pandas as pd
import yfinance as yf
from bs4 import BeautifulSoup

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont

try:
    import yt_dlp
except Exception as e:
    yt_dlp = None
    print("yt-dlpの読み込みに失敗しました。YouTube取得はスキップされます:", e)

# ===== ログ設定 =====
today = datetime.now().strftime("%Y-%m-%d")
log_path = os.path.join(LOG_DIR, f"daily_{today}.log")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_path, encoding="utf-8"),
        logging.StreamHandler()
    ],
)
logger = logging.getLogger("us_stock_ai_report_mvp")
logger.info("処理開始")

# ===== 取得対象 =====
# 初回用に、なるべく編集不要な検索クエリを用意しています。
# 精度を上げる段階で、ここに特定チャンネルURLや動画URLを追加します。
YOUTUBE_SEARCHES = [
    {"speaker": "Jim Cramer", "query": "ytsearch4:Jim Cramer Mad Money stock picks today CNBC"},
    {"speaker": "Jim Cramer Commentary", "query": "ytsearch3:米国株 ジム クレイマー 解説 最新"},
    {"speaker": "Ray Dalio", "query": "ytsearch3:Ray Dalio markets economy debt cycle latest"},
    {"speaker": "Scott Galloway", "query": "ytsearch3:Scott Galloway AI big tech stocks latest"},
    {"speaker": "Scott Galloway", "query": "ytsearch2:Prof G Markets Scott Galloway latest"},
]

RSS_FEEDS = [
    {
        "name": "Yahoo Finance Big Tech Headlines",
        "speaker": "Market News",
        "url": "https://feeds.finance.yahoo.com/rss/2.0/headline?s=AAPL,MSFT,NVDA,AMD,AVGO,META,GOOGL,AMZN,TSLA,PLTR,SMCI,TSM,ORCL,CRM,NFLX,COST,LLY&region=US&lang=en-US",
    }
]

# ===== 銘柄辞書 =====
# 最初は主要銘柄中心。あとで増やせます。
COMPANY_ALIASES = {
    "AAPL": ["Apple", "アップル"],
    "MSFT": ["Microsoft", "マイクロソフト"],
    "NVDA": ["Nvidia", "NVIDIA", "エヌビディア", "エヌビディア"],
    "AMD": ["AMD", "Advanced Micro Devices"],
    "AVGO": ["Broadcom", "ブロードコム"],
    "META": ["Meta", "Facebook", "メタ"],
    "GOOGL": ["Google", "Alphabet", "グーグル", "アルファベット"],
    "AMZN": ["Amazon", "アマゾン"],
    "TSLA": ["Tesla", "テスラ"],
    "PLTR": ["Palantir", "パランティア"],
    "IONQ": ["IonQ", "IONQ"],
    "SMCI": ["Super Micro", "Supermicro", "SMCI"],
    "TSM": ["TSMC", "Taiwan Semiconductor", "台湾セミコンダクター"],
    "CRM": ["Salesforce", "セールスフォース"],
    "ORCL": ["Oracle", "オラクル"],
    "NFLX": ["Netflix", "ネットフリックス"],
    "COST": ["Costco", "コストコ"],
    "LLY": ["Eli Lilly", "Lilly", "イーライリリー"],
    "RTX": ["RTX", "Raytheon"],
    "VALE": ["Vale", "ヴァーレ"],
    "JPM": ["JPMorgan", "JP Morgan", "JPMorgan Chase"],
    "BAC": ["Bank of America", "バンク・オブ・アメリカ"],
    "XOM": ["Exxon", "Exxon Mobil"],
    "CVX": ["Chevron", "シェブロン"],
    "DIS": ["Disney", "ディズニー"],
    "UBER": ["Uber", "ウーバー"],
    "SHOP": ["Shopify", "ショッピファイ"],
    "SNOW": ["Snowflake", "スノーフレイク"],
    "COIN": ["Coinbase", "コインベース"],
    "MSTR": ["MicroStrategy", "Strategy", "マイクロストラテジー"],
}

BULLISH_WORDS = [
    "buy", "buying", "bought", "own", "like", "love", "bullish", "winner",
    "outperform", "upside", "opportunity", "growth", "strong", "accelerating",
    "benefit", "beneficiary", "upgrade", "beats", "raises", "best", "compounder",
    "long-term", "ai", "artificial intelligence", "data center", "cloud",
    "買い", "強気", "上昇", "成長", "有望", "注目", "期待", "好決算", "追い風"
]
BEARISH_WORDS = [
    "sell", "selling", "avoid", "don't buy", "do not buy", "bearish", "risk",
    "overvalued", "expensive", "recession", "debt", "weak", "downgrade", "miss",
    "slowing", "decline", "bubble", "fraud", "regulation", "lawsuit",
    "売り", "避け", "弱気", "下落", "リスク", "割高", "景気後退", "鈍化", "赤字", "規制", "危険"
]
RISK_WORDS = [
    "risk", "overvalued", "expensive", "debt", "recession", "regulation", "competition",
    "lawsuit", "slowing", "margin pressure", "bubble",
    "リスク", "割高", "債務", "景気後退", "規制", "競争", "訴訟", "鈍化", "過熱"
]
THEME_WORDS = [
    "ai", "artificial intelligence", "semiconductor", "data center", "cloud", "gpu",
    "cybersecurity", "software", "automation", "robotics", "energy", "defense",
    "人工知能", "半導体", "データセンター", "クラウド", "防衛", "エネルギー"
]

SPEAKER_WEIGHT = {
    "Jim Cramer": 0.25,
    "Jim Cramer Commentary": 0.20,
    "Ray Dalio": 0.18,
    "Scott Galloway": 0.22,
    "Market News": 0.15,
}

# ===== 共通関数 =====
def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def clean_text(text):
    if not text:
        return ""
    text = html.unescape(str(text))
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def truncate(text, n=1800):
    text = clean_text(text)
    return text[:n] + ("..." if len(text) > n else "")

def count_keywords(text, words):
    low = text.lower()
    return sum(1 for w in words if w.lower() in low)

def classify_stance(text):
    b = count_keywords(text, BULLISH_WORDS)
    r = count_keywords(text, BEARISH_WORDS)
    if b == 0 and r == 0:
        return "中立/不明", 0.0
    raw = (b - r) / max(1, b + r)
    if raw > 0.25:
        return "強気", raw
    if raw < -0.25:
        return "弱気/注意", raw
    return "中立", raw

def parse_json3_caption(text):
    try:
        data = json.loads(text)
        parts = []
        for ev in data.get("events", []):
            for seg in ev.get("segs", []) or []:
                if "utf8" in seg:
                    parts.append(seg["utf8"])
        return clean_text(" ".join(parts))
    except Exception:
        return ""

def clean_vtt_caption(text):
    lines = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("WEBVTT") or line.startswith("Kind:") or line.startswith("Language:"):
            continue
        if "-->" in line:
            continue
        if re.match(r"^\d+$", line):
            continue
        line = re.sub(r"<[^>]+>", "", line)
        line = re.sub(r"\{.*?\}", "", line)
        lines.append(line)
    # 連続重複を軽く除去
    dedup = []
    prev = None
    for line in lines:
        if line != prev:
            dedup.append(line)
        prev = line
    return clean_text(" ".join(dedup))

def fetch_caption_from_entry(entry):
    """yt-dlpで得た字幕URLから短いテキストを取得。音声ダウンロードはしません。"""
    candidates = []
    for bucket_name in ["subtitles", "automatic_captions"]:
        bucket = entry.get(bucket_name) or {}
        # 日本語・英語を優先
        langs = ["ja", "en", "en-US", "en-orig"]
        langs += [k for k in bucket.keys() if k not in langs]
        for lang in langs:
            tracks = bucket.get(lang) or []
            for track in tracks:
                url = track.get("url")
                ext = track.get("ext", "")
                if url:
                    candidates.append((lang, ext, url))
    # json3を優先、次にvtt
    candidates.sort(key=lambda x: 0 if "json3" in x[1] else 1)
    for lang, ext, url in candidates[:5]:
        try:
            resp = requests.get(url, timeout=20)
            if resp.status_code != 200:
                continue
            txt = resp.text
            parsed = parse_json3_caption(txt) if "json" in ext else clean_vtt_caption(txt)
            if len(parsed) > 200:
                return parsed[:8000]
        except Exception:
            continue
    return ""

def collect_youtube_items():
    items = []
    if yt_dlp is None:
        logger.warning("yt-dlpが使えないためYouTube取得をスキップします。")
        return items

    ydl_opts = {
        "quiet": True,
        "skip_download": True,
        "noplaylist": True,
        "extract_flat": False,
        "ignoreerrors": True,
        "socket_timeout": 25,
        "nocheckcertificate": True,
    }

    for src in YOUTUBE_SEARCHES:
        speaker = src["speaker"]
        query = src["query"]
        logger.info(f"YouTube検索: {speaker} / {query}")
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(query, download=False)
            entries = info.get("entries", []) if isinstance(info, dict) else []
            for e in entries:
                if not e:
                    continue
                title = clean_text(e.get("title", ""))
                url = e.get("webpage_url") or e.get("original_url") or e.get("url") or ""
                description = clean_text(e.get("description", ""))
                upload_date = e.get("upload_date", "")
                caption = fetch_caption_from_entry(e)
                text = clean_text(" ".join([title, description, caption]))
                if len(text) < 40:
                    text = clean_text(" ".join([title, description]))
                items.append({
                    "source_type": "youtube",
                    "speaker": speaker,
                    "title": title,
                    "url": url,
                    "published": upload_date,
                    "text": text,
                    "caption_chars": len(caption),
                })
        except Exception as ex:
            logger.warning(f"YouTube取得失敗: {speaker} / {ex}")
    logger.info(f"YouTube取得件数: {len(items)}")
    return items

def collect_rss_items():
    items = []
    for src in RSS_FEEDS:
        try:
            logger.info(f"RSS取得: {src['name']}")
            feed = feedparser.parse(src["url"])
            for e in feed.entries[:10]:
                title = clean_text(getattr(e, "title", ""))
                summary = clean_text(getattr(e, "summary", ""))
                link = getattr(e, "link", "")
                published = getattr(e, "published", "")
                text = clean_text(f"{title} {summary}")
                if len(text) < 20:
                    continue
                items.append({
                    "source_type": "rss",
                    "speaker": src["speaker"],
                    "title": title,
                    "url": link,
                    "published": published,
                    "text": text,
                    "caption_chars": 0,
                })
        except Exception as ex:
            logger.warning(f"RSS取得失敗: {src.get('name')}: {ex}")
    logger.info(f"RSS取得件数: {len(items)}")
    return items

def find_ticker_mentions(item):
    text = item["text"]
    low = text.lower()
    mentions = []
    for ticker, aliases in COMPANY_ALIASES.items():
        found = False
        patterns = [ticker] + aliases
        match_pos = -1
        matched = ""
        for p in patterns:
            if p == ticker:
                # ティッカーは大文字単語または $NVDA 形式を優先
                m = re.search(rf"(?<![A-Z])\\${ticker}\\b|\\b{ticker}\\b", text, flags=re.IGNORECASE)
            else:
                m = re.search(re.escape(p), text, flags=re.IGNORECASE)
            if m:
                found = True
                match_pos = m.start()
                matched = p
                break
        if not found:
            continue
        start = max(0, match_pos - 350)
        end = min(len(text), match_pos + 550)
        snippet = clean_text(text[start:end])
        stance, stance_score = classify_stance(snippet)
        risk_hits = count_keywords(snippet, RISK_WORDS)
        theme_hits = count_keywords(snippet, THEME_WORDS)
        mentions.append({
            "ticker": ticker,
            "company": aliases[0],
            "speaker": item["speaker"],
            "source_type": item["source_type"],
            "title": item["title"],
            "url": item["url"],
            "published": item.get("published", ""),
            "snippet": truncate(snippet, 700),
            "stance": stance,
            "stance_score": float(stance_score),
            "risk_hits": risk_hits,
            "theme_hits": theme_hits,
        })
    return mentions

def get_market_data(ticker):
    result = {
        "ticker": ticker,
        "price": None,
        "return_3m": None,
        "return_6m": None,
        "return_1y": None,
        "market_cap": None,
        "pe": None,
        "data_ok": False,
    }
    try:
        tk = yf.Ticker(ticker)
        hist = tk.history(period="1y", auto_adjust=True)
        if hist is not None and len(hist) > 5:
            closes = hist["Close"].dropna()
            if len(closes) > 5:
                price = float(closes.iloc[-1])
                result["price"] = price
                def ret(days):
                    if len(closes) > days:
                        return float(closes.iloc[-1] / closes.iloc[-days] - 1)
                    return None
                result["return_3m"] = ret(63)
                result["return_6m"] = ret(126)
                result["return_1y"] = ret(252)
                result["data_ok"] = True
        try:
            fi = tk.fast_info
            mc = fi.get("market_cap", None) if hasattr(fi, "get") else None
            if mc:
                result["market_cap"] = float(mc)
        except Exception:
            pass
        # infoは遅いことがあるので失敗しても続行
        try:
            info = tk.get_info()
            if not result["market_cap"] and info.get("marketCap"):
                result["market_cap"] = float(info.get("marketCap"))
            if info.get("trailingPE"):
                result["pe"] = float(info.get("trailingPE"))
        except Exception:
            pass
    except Exception as ex:
        logger.warning(f"市場データ取得失敗: {ticker} / {ex}")
    return result

def pct(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "N/A"
    return f"{x*100:.1f}%"

def money(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "N/A"
    if x >= 1_000_000_000_000:
        return f"${x/1_000_000_000_000:.2f}T"
    if x >= 1_000_000_000:
        return f"${x/1_000_000_000:.1f}B"
    if x >= 1_000_000:
        return f"${x/1_000_000:.1f}M"
    return f"${x:,.0f}"

def score_tickers(mentions):
    grouped = defaultdict(list)
    for m in mentions:
        grouped[m["ticker"]].append(m)

    rows = []
    for ticker, ms in grouped.items():
        md = get_market_data(ticker)

        weighted_scores = []
        for m in ms:
            w = SPEAKER_WEIGHT.get(m["speaker"], 0.10)
            weighted_scores.append(m["stance_score"] * w)
        total_w = sum(SPEAKER_WEIGHT.get(m["speaker"], 0.10) for m in ms)
        sentiment = sum(weighted_scores) / max(0.01, total_w)

        source_count = len(set(m["url"] for m in ms if m.get("url")))
        speakers = sorted(set(m["speaker"] for m in ms))
        risk_hits = sum(m["risk_hits"] for m in ms)
        theme_hits = sum(m["theme_hits"] for m in ms)

        momentum_bonus = 0
        if md["return_6m"] is not None:
            if md["return_6m"] > 0.35:
                momentum_bonus += 10
            elif md["return_6m"] > 0.10:
                momentum_bonus += 6
            elif md["return_6m"] < -0.25:
                momentum_bonus -= 8
            elif md["return_6m"] < -0.10:
                momentum_bonus -= 4

        theme_bonus = min(8, theme_hits * 1.5)
        source_bonus = min(10, source_count * 3)
        data_bonus = 4 if md["data_ok"] else -5

        valuation_penalty = 0
        if md["pe"] is not None:
            if md["pe"] > 100:
                valuation_penalty += 8
            elif md["pe"] > 60:
                valuation_penalty += 5

        risk_penalty = min(15, risk_hits * 2)

        upside_score = 50 + sentiment * 28 + momentum_bonus + theme_bonus + source_bonus + data_bonus - valuation_penalty - risk_penalty * 0.45
        avoid_score = 45 - sentiment * 22 - momentum_bonus * 0.5 + risk_penalty + valuation_penalty - theme_bonus * 0.3
        upside_score = int(max(0, min(100, round(upside_score))))
        avoid_score = int(max(0, min(100, round(avoid_score))))

        # Cramerだけに依存していないか
        non_cramer = [s for s in speakers if "Cramer" not in s]
        cramer_dependency = "高い" if len(non_cramer) == 0 else "低〜中"

        main_reason = []
        if sentiment > 0.20:
            main_reason.append("取得ソース上の言及は強気寄り")
        elif sentiment < -0.20:
            main_reason.append("取得ソース上の言及は弱気・注意寄り")
        else:
            main_reason.append("取得ソース上の言及は中立寄り")
        if theme_hits > 0:
            main_reason.append("AI・半導体・クラウド等の成長テーマへの言及あり")
        if md["return_6m"] is not None:
            main_reason.append(f"6カ月リターン {pct(md['return_6m'])}")
        if source_count >= 2:
            main_reason.append("複数ソースで言及")
        if cramer_dependency == "高い":
            main_reason.append("Cramer関連ソースへの依存度が高い点に注意")

        main_risk = []
        if risk_hits > 0:
            main_risk.append("リスク・割高・規制・鈍化などの注意語が含まれる")
        if valuation_penalty > 0:
            main_risk.append("PERが高くバリュエーション面の注意あり")
        if md["return_6m"] is not None and md["return_6m"] > 0.5:
            main_risk.append("短期的に上昇済みで過熱感に注意")
        if not main_risk:
            main_risk.append("定量・定性情報がまだ限定的")

        rows.append({
            "ticker": ticker,
            "company": COMPANY_ALIASES[ticker][0],
            "upside_score": upside_score,
            "avoid_score": avoid_score,
            "sentiment": sentiment,
            "speakers": ", ".join(speakers),
            "source_count": source_count,
            "cramer_dependency": cramer_dependency,
            "reason": "。".join(main_reason),
            "risk": "。".join(main_risk),
            "market": md,
            "mentions": ms,
        })
    return rows

def fallback_rows_if_needed(rows):
    """言及が少なすぎる場合でもPDFを完成させるため、市場データだけの参考欄を作る。"""
    if len(rows) >= 3:
        return rows, False

    logger.warning("銘柄言及が少ないため、市場データのみの参考候補を追加します。")
    existing = {r["ticker"] for r in rows}
    fallback_tickers = ["NVDA", "MSFT", "AAPL", "GOOGL", "AMZN", "META", "AVGO", "TSLA", "PLTR"]
    for ticker in fallback_tickers:
        if ticker in existing:
            continue
        md = get_market_data(ticker)
        momentum_bonus = 0
        if md["return_6m"] is not None:
            if md["return_6m"] > 0.35: momentum_bonus = 10
            elif md["return_6m"] > 0.10: momentum_bonus = 6
            elif md["return_6m"] < -0.25: momentum_bonus = -8
        upside = int(max(0, min(100, round(50 + momentum_bonus + (4 if md["data_ok"] else -5)))))
        avoid = int(max(0, min(100, round(45 - momentum_bonus * 0.5))))
        rows.append({
            "ticker": ticker,
            "company": COMPANY_ALIASES[ticker][0],
            "upside_score": upside,
            "avoid_score": avoid,
            "sentiment": 0,
            "speakers": "市場データのみ",
            "source_count": 0,
            "cramer_dependency": "該当なし",
            "reason": "今回の公開ソースから十分な個別言及を取得できなかったため、市場データ確認用の参考候補として掲載。",
            "risk": "著名人発言・一次情報との照合が不足。投資判断には追加確認が必要。",
            "market": md,
            "mentions": [],
        })
        if len(rows) >= 6:
            break
    return rows, True

def build_pdf(items, rows, fallback_used):
    pdfmetrics.registerFont(UnicodeCIDFont("HeiseiKakuGo-W5"))
    pdfmetrics.registerFont(UnicodeCIDFont("HeiseiMin-W3"))

    report_date = datetime.now().strftime("%Y-%m-%d")
    pdf_path = os.path.join(REPORT_DIR, f"us_stock_ai_report_{report_date}.pdf")

    doc = SimpleDocTemplate(
        pdf_path,
        pagesize=A4,
        rightMargin=14*mm,
        leftMargin=14*mm,
        topMargin=14*mm,
        bottomMargin=14*mm,
    )

    styles = getSampleStyleSheet()
    base = ParagraphStyle(
        "JPBase",
        parent=styles["Normal"],
        fontName="HeiseiKakuGo-W5",
        fontSize=9.2,
        leading=13,
        spaceAfter=5,
    )
    small = ParagraphStyle(
        "JPSmall",
        parent=base,
        fontSize=7.4,
        leading=10,
    )
    title = ParagraphStyle(
        "JPTitle",
        parent=styles["Title"],
        fontName="HeiseiKakuGo-W5",
        fontSize=18,
        leading=24,
        spaceAfter=12,
    )
    h1 = ParagraphStyle(
        "JPH1",
        parent=styles["Heading1"],
        fontName="HeiseiKakuGo-W5",
        fontSize=14,
        leading=18,
        spaceBefore=8,
        spaceAfter=8,
    )
    h2 = ParagraphStyle(
        "JPH2",
        parent=styles["Heading2"],
        fontName="HeiseiKakuGo-W5",
        fontSize=11.5,
        leading=15,
        spaceBefore=8,
        spaceAfter=5,
    )

    def P(text, style=base):
        return Paragraph(escape(str(text)).replace("\n", "<br/>"), style)

    def cell(text):
        return Paragraph(escape(str(text)), small)

    story = []
    story.append(P("米国株AI投資調査レポート", title))
    story.append(P(f"作成日時: {now_str()}"))
    story.append(P("対象: Jim Cramer、Ray Dalio、Scott Galloway関連の公開YouTube/RSS情報、およびyfinanceの市場データ"))
    story.append(P("位置づけ: 本レポートは投資助言ではなく、公開情報を整理した調査資料です。"))
    story.append(Spacer(1, 8))

    story.append(P("1. 本日の結論", h1))
    top = sorted(rows, key=lambda r: r["upside_score"], reverse=True)[:5]
    avoid = sorted(rows, key=lambda r: r["avoid_score"], reverse=True)[:5]

    if fallback_used:
        story.append(P("注意: 今回は取得ソースから個別銘柄の十分な言及を取り切れなかったため、一部に市場データのみの参考候補を含めています。", base))

    story.append(P("大幅上昇の可能性がある候補（調査候補）:", h2))
    for r in top:
        story.append(P(f"{r['ticker']} / {r['company']}：上昇期待スコア {r['upside_score']}。{r['reason']}", base))

    story.append(P("注意・回避候補:", h2))
    for r in avoid:
        story.append(P(f"{r['ticker']} / {r['company']}：注意スコア {r['avoid_score']}。{r['risk']}", base))

    story.append(PageBreak())

    story.append(P("2. 上昇候補ランキング", h1))
    data = [[
        cell("Ticker"), cell("会社名"), cell("株価"), cell("6カ月"), cell("上昇"), cell("注意"), cell("根拠"), cell("リスク")
    ]]
    for r in top:
        md = r["market"]
        data.append([
            cell(r["ticker"]),
            cell(r["company"]),
            cell(f"${md['price']:.2f}" if md["price"] else "N/A"),
            cell(pct(md["return_6m"])),
            cell(r["upside_score"]),
            cell(r["avoid_score"]),
            cell(r["reason"]),
            cell(r["risk"]),
        ])
    table = Table(data, colWidths=[15*mm, 22*mm, 18*mm, 17*mm, 14*mm, 14*mm, 48*mm, 42*mm])
    table.setStyle(TableStyle([
        ("FONTNAME", (0,0), (-1,-1), "HeiseiKakuGo-W5"),
        ("FONTSIZE", (0,0), (-1,-1), 7),
        ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
        ("GRID", (0,0), (-1,-1), 0.25, colors.grey),
        ("VALIGN", (0,0), (-1,-1), "TOP"),
    ]))
    story.append(table)

    story.append(Spacer(1, 10))
    story.append(P("3. 注意候補ランキング", h1))
    data = [[
        cell("Ticker"), cell("会社名"), cell("株価"), cell("6カ月"), cell("上昇"), cell("注意"), cell("注意理由"), cell("Cramer依存")
    ]]
    for r in avoid:
        md = r["market"]
        data.append([
            cell(r["ticker"]),
            cell(r["company"]),
            cell(f"${md['price']:.2f}" if md["price"] else "N/A"),
            cell(pct(md["return_6m"])),
            cell(r["upside_score"]),
            cell(r["avoid_score"]),
            cell(r["risk"]),
            cell(r["cramer_dependency"]),
        ])
    table = Table(data, colWidths=[15*mm, 22*mm, 18*mm, 17*mm, 14*mm, 14*mm, 67*mm, 23*mm])
    table.setStyle(TableStyle([
        ("FONTNAME", (0,0), (-1,-1), "HeiseiKakuGo-W5"),
        ("FONTSIZE", (0,0), (-1,-1), 7),
        ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
        ("GRID", (0,0), (-1,-1), 0.25, colors.grey),
        ("VALIGN", (0,0), (-1,-1), "TOP"),
    ]))
    story.append(table)

    story.append(PageBreak())

    story.append(P("4. 発言者別の取得情報", h1))
    for speaker in ["Jim Cramer", "Jim Cramer Commentary", "Ray Dalio", "Scott Galloway", "Market News"]:
        speaker_items = [i for i in items if i["speaker"] == speaker]
        if not speaker_items:
            continue
        story.append(P(speaker, h2))
        for i in speaker_items[:5]:
            cap_note = f"字幕文字数: {i.get('caption_chars', 0)}"
            story.append(P(f"・{i['title']}（{cap_note}）", base))
            story.append(P(f"  URL: {i['url']}", small))
            story.append(P(f"  要約用テキスト抜粋: {truncate(i['text'], 450)}", small))

    story.append(PageBreak())
    story.append(P("5. 銘柄別メモと出典", h1))
    for r in sorted(rows, key=lambda r: r["upside_score"], reverse=True):
        story.append(P(f"{r['ticker']} / {r['company']}", h2))
        md = r["market"]
        story.append(P(f"株価: {f'${md['price']:.2f}' if md['price'] else 'N/A'} / 時価総額: {money(md['market_cap'])} / PER: {md['pe'] if md['pe'] else 'N/A'} / 6カ月リターン: {pct(md['return_6m'])}"))
        story.append(P(f"上昇期待スコア: {r['upside_score']} / 注意スコア: {r['avoid_score']} / Cramer依存度: {r['cramer_dependency']}"))
        story.append(P(f"根拠: {r['reason']}"))
        story.append(P(f"リスク: {r['risk']}"))
        if r["mentions"]:
            for m in r["mentions"][:3]:
                story.append(P(f"出典: {m['speaker']} / {m['title']} / {m['url']}", small))
                story.append(P(f"抜粋: {m['snippet']}", small))
        else:
            story.append(P("出典: 今回は個別発言の十分な取得なし。市場データのみの参考表示。", small))
        story.append(Spacer(1, 5))

    story.append(PageBreak())
    story.append(P("6. 免責事項", h1))
    story.append(P("本レポートは公開情報をもとにAIが自動生成した調査資料であり、投資助言ではありません。最終的な投資判断は利用者自身の責任で行ってください。記載内容には誤り、遅延、不完全な情報が含まれる可能性があります。", base))
    story.append(P("また、Jim Cramer、Ray Dalio、Scott Galloway等の発言・見解は、取得できた公開情報の範囲で要約・分類したものであり、本人の投資助言や公式な推奨を意味しません。", base))
    story.append(P(f"ログ保存先: {log_path}", small))

    doc.build(story)
    return pdf_path

def send_email_with_pdf(pdf_path):
    if not SEND_EMAIL:
        logger.info("SEND_EMAIL=Falseのためメール送信をスキップしました。")
        return False

    if not EMAIL_TO or not SMTP_USER or not SMTP_PASSWORD:
        logger.warning("メール設定が不足しているため送信をスキップしました。")
        return False

    msg = EmailMessage()
    msg["Subject"] = f"【米国株AIレポート】{datetime.now().strftime('%Y-%m-%d')}版"
    msg["From"] = EMAIL_FROM
    msg["To"] = EMAIL_TO
    msg.set_content(
        "米国株AI投資調査レポートを添付します。\n\n"
        "本レポートは公開情報をもとにAIが自動生成した調査資料であり、投資助言ではありません。\n"
        "最終的な投資判断は利用者自身の責任で行ってください。\n"
    )
    with open(pdf_path, "rb") as f:
        msg.add_attachment(
            f.read(),
            maintype="application",
            subtype="pdf",
            filename=os.path.basename(pdf_path),
        )

    with smtplib.SMTP("smtp.gmail.com", 587, timeout=40) as server:
        server.starttls()
        server.login(SMTP_USER, SMTP_PASSWORD)
        server.send_message(msg)

    logger.info(f"メール送信完了: {EMAIL_TO}")
    return True

# ===== 実行 =====
try:
    logger.info("情報収集を開始します。")
    items = []
    items.extend(collect_youtube_items())
    items.extend(collect_rss_items())

    # 重複URLを軽く除去
    seen = set()
    unique_items = []
    for item in items:
        key = item.get("url") or item.get("title")
        if key in seen:
            continue
        seen.add(key)
        unique_items.append(item)
    items = unique_items

    logger.info(f"取得アイテム合計: {len(items)}")

    logger.info("銘柄抽出を開始します。")
    mentions = []
    for item in items:
        mentions.extend(find_ticker_mentions(item))

    logger.info(f"抽出された銘柄言及数: {len(mentions)}")

    logger.info("スコアリングを開始します。")
    rows = score_tickers(mentions)
    rows, fallback_used = fallback_rows_if_needed(rows)

    logger.info("PDF生成を開始します。")
    pdf_path = build_pdf(items, rows, fallback_used)
    logger.info(f"PDF保存完了: {pdf_path}")

    logger.info("メール送信を開始します。")
    sent = send_email_with_pdf(pdf_path)

    print("\n===== 完了 =====")
    print("PDF保存先:", pdf_path)
    print("ログ保存先:", log_path)
    print("メール送信:", "完了" if sent else "スキップまたは未送信")
    print("\nGoogle Drive の ai-investment-agent/reports フォルダを確認してください。")

except Exception as e:
    logger.error("処理中にエラーが発生しました。")
    logger.error(traceback.format_exc())
    print("\n===== エラー =====")
    print("ログを確認してください:", log_path)
    print(str(e))

## 次にやること

1回目が成功したら、次は検索クエリや対象銘柄を増やします。  
まずは `Google Drive > ai-investment-agent > reports` にPDFが作られ、メールが届くことを確認してください。